# 3-Way Sparse EBM — Thesis Benchmark Suite
## Notebook Map

| Block | Title | Key Output |
|-------|-------|-----------|
| 0 | Global Config | `DEBUG_MODE`, seeds, paths |
| 1 | Installs & Imports | libraries |
| 2 | Core Algorithm | `Sparse3WayEBM`, `Sparse3WayEBMWithBagging` |
| 3 | Model Wrappers | `supports_found_rate()`, `avg_bag_term_count_` |
| 4 | Dataset Suite | synthetic + real loaders |
| 5 | Splits | reproducible train/test indices |
| 6 | Hyperparameter Tuning | per-dataset, Bischl 2023 search spaces |
| 7 | Build Models | `build_models_for_dataset()` |
| 8 | Benchmark Engine | `run_benchmark()`, `summarize()` |
| 9 | Standard Synthetic | H1 baseline |
| 10 | Modified Synthetic | H1 + H2 (found_rate) |
| 11 | BIC Sweep | H3 (sparsity) |
| 12 | FAST-3D Recovery | H2 (Recall@K) |
| 13 | Real Datasets | H4 (competitiveness) |
| 14 | Ablation: Cyclic Refit | design validation |
| 15 | Smoke Test | quick debug |
| 16 | Download | zip outputs |

**Run order:** 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → then any of 9-14 independently.

### Changelog
- `found_rate`: NA for XGBoost/RF (cannot extract term structure)
- Per-dataset hyperparameter tuning (Bischl et al. 2023 search spaces)
- Real datasets: 5 from OpenML-CTR23 (Fischer et al. 2023)
- Term output logging: `collect_terms=True` in `run_benchmark()`
- FAST-3D 2-step pre-filter for high-d datasets
- BIC k_params cap: `min(n_bins-1, 64)` for main effects
- `avg_bag_term_count_` for correct H3 term-count metric


## BLOCK 0 — Global Config
Change `DEBUG_MODE = True` for fast smoke testing (small n, 1 seed).

In [ ]:
import time as _time

# ── EDIT HERE ONLY ────────────────────────────────────────────────────────────
DEBUG_MODE   = False        # True → fast run (n=300, seeds=[0])
GLOBAL_SEEDS = [0,1,2]      # thesis: 3 seeds
N_JOBS       = 4            # parallel jobs for joblib
TEST_SIZE    = 0.2
SYNTH_N      = 2000         # synthetic dataset size
BIC_SCALES   = [0.0, 0.1, 0.25, 0.5, 0.75, 1]  
TUNING_N_SPLITS = 3         # inner CV folds for hyperparameter search
# ─────────────────────────────────────────────────────────────────────────────

if DEBUG_MODE:
    GLOBAL_SEEDS = [0]
    SYNTH_N      = 300
    print("[DEBUG MODE] seeds=[0], n=300")

import os, json, warnings
RUN_ID      = _time.strftime("%Y%m%d-%H%M%S")
OUTPUT_ROOT = os.path.join("outputs", RUN_ID)
DIRS = {
    "tables": os.path.join(OUTPUT_ROOT, "tables"),
    "figs":   os.path.join(OUTPUT_ROOT, "figs"),
    "logs":   os.path.join(OUTPUT_ROOT, "logs"),
    "splits": os.path.join(OUTPUT_ROOT, "splits"),
    "hparam": os.path.join(OUTPUT_ROOT, "hparam"),
    "models": os.path.join(OUTPUT_ROOT, "models"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(f"RUN_ID: {RUN_ID}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")


## BLOCK 1 — Installs & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools, copy, warnings
from collections import Counter
from math import comb
import joblib

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import make_friedman1, load_diabetes, fetch_openml
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# Optional dependencies
HAS_INTERPRET = False
try:
    from interpret.glassbox import ExplainableBoostingRegressor
    HAS_INTERPRET = True
except Exception:
    print("interpretML not found — VanillaEBM will be skipped.")

HAS_XGBOOST = False
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    print("xgboost not found — XGBoost will be skipped.")

import sklearn
print(f"sklearn {sklearn.__version__} | interpret={HAS_INTERPRET} | xgboost={HAS_XGBOOST}")


In [ ]:
# ── Utility helpers ──────────────────────────────────────────────────────────
plt.rcParams.update({"figure.figsize":(9,5), "axes.titlesize":13,
                     "axes.labelsize":11, "xtick.labelsize":9, "ytick.labelsize":9})

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def export_fig(fig, name, tight=True):
    if tight: fig.tight_layout()
    for ext in ["pdf", "png"]:
        p = os.path.join(DIRS["figs"], f"{name}.{ext}")
        fig.savefig(p, bbox_inches="tight", dpi=150)
    print(f"  fig saved: {name}")

def export_table(df, name, floatfmt="%.4f"):
    csv_p = os.path.join(DIRS["tables"], f"{name}.csv")
    tex_p = os.path.join(DIRS["tables"], f"{name}.tex")
    df.to_csv(csv_p, index=False)
    with open(tex_p, "w") as f:
        f.write(df.to_latex(index=False,
            float_format=lambda x: floatfmt % x if isinstance(x, (float, np.floating)) else str(x)))
    print(f"  table saved: {name}")

def rmse(y_true, y_pred): return float(np.sqrt(mean_squared_error(y_true, y_pred)))
def mae(y_true, y_pred):  return float(mean_absolute_error(y_true, y_pred))
def r2(y_true, y_pred):   return float(r2_score(y_true, y_pred))

save_json({"RUN_ID": RUN_ID, "DEBUG_MODE": DEBUG_MODE,
           "GLOBAL_SEEDS": GLOBAL_SEEDS, "HAS_INTERPRET": HAS_INTERPRET,
           "HAS_XGBOOST": HAS_XGBOOST},
          os.path.join(DIRS["logs"], "run_config.json"))
print("Helpers ready.")


## BLOCK 2 — Core Algorithm
**DO NOT EDIT** — `Sparse3WayEBM` and `Sparse3WayEBMWithBagging`.



In [ ]:
class Sparse3WayEBM(BaseEstimator, RegressorMixin):
    """
    Sparse 3-Way EBM with cyclic refitting and BIC-based backward pruning.
    Supports main effects (degree=1), pairwise (degree=2), and triple (degree=3) interactions.

    Key parameters
    --------------
    max_interaction_degree : int
        Maximum interaction order (2=pairwise, 3=triple).
    interactions : int
        Number of interaction terms to select per degree.
    penalty_scale : float
        BIC penalty multiplier. 0.0 = no pruning.
    early_stopping_rounds : int
        OOB early stopping patience (checked every epoch).
        0 = disabled. Requires OOB data passed via _oob_X_ / _oob_y_raw_.
    """
    def __init__(self,
                 learning_rate=0.01,
                 n_estimators=100,
                 max_bins=32,
                 interactions=0,
                 max_interaction_degree=2,
                 penalty_scale=0.0,
                 selection_criterion="bic",
                 random_state=42,
                 enable_joint_refit=True,
                 early_stopping_rounds=200):
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_bins = max_bins
        self.interactions = interactions
        self.max_interaction_degree = max_interaction_degree
        self.penalty_scale = penalty_scale
        self.selection_criterion = selection_criterion
        self.random_state = random_state
        self.enable_joint_refit = enable_joint_refit
        self.early_stopping_rounds = early_stopping_rounds

        # model state
        self.term_features_ = []
        self.additive_terms_ = []
        self.term_names_     = []
        self.global_mean_    = 0.0
        self.y_mean_         = 0.0
        self.y_std_          = 1.0
        self.bins            = []
        self.best_iteration_ = None   # tracks early stopping epoch

    # ── Discretisation ───────────────────────────────────────────────────────
    def _discretize(self, X):
        bins = []
        X_binned = np.zeros(X.shape, dtype=int)
        for i in range(X.shape[1]):
            unique = np.unique(X[:, i])
            if len(unique) <= self.max_bins:
                edges = np.concatenate([[-np.inf], np.sort(unique)[1:], [np.inf]])
            else:
                edges = np.percentile(X[:, i], np.linspace(0, 100, self.max_bins + 1))
                edges[0], edges[-1] = -np.inf, np.inf
            bins.append(edges)
            X_binned[:, i] = np.clip(np.digitize(X[:, i], edges) - 1,
                                     0, len(edges) - 2)
        return X_binned, bins

    def _apply_bins(self, X):
        X_binned = np.zeros(X.shape, dtype=int)
        for i in range(X.shape[1]):
            X_binned[:, i] = np.clip(np.digitize(X[:, i], self.bins[i]) - 1,
                                     0, len(self.bins[i]) - 2)
        return X_binned

    # ── Stump learner (main effects) ─────────────────────────────────────────
    def _fast_stump_learner(self, X_col, residuals, n_bins):
        sum_grad = np.bincount(X_col, weights=residuals, minlength=n_bins)
        count    = np.bincount(X_col, minlength=n_bins)
        with np.errstate(divide="ignore", invalid="ignore"):
            update = sum_grad / count
        update[np.isnan(update)] = 0.0
        return update

    # ── FAST-3D interaction screening ───────────
    def _find_interactions(self, X, residuals, degree):
        """
        Exhaustive FAST-style interaction screening.
        Scores all C(d, degree) candidates using residual-weighted
        histogram statistics and returns the top-K by score.

        Ref: Lou et al. (2013), Section 2.2

        added pre filter for d>20  
        """
        d = X.shape[1]
    
        # ── added added pre filter for d>20 ────────────────────────────────────────────────
        if d > 20 and degree >= 3:
            feat_agg = np.zeros(d)
            for i, j in itertools.combinations(range(d), 2):
                X_sub = X[:, [i, j]]
                H_s, _ = np.histogramdd(X_sub, bins=8, weights=residuals)
                H_c, _ = np.histogramdd(X_sub, bins=8)
                with np.errstate(divide='ignore', invalid='ignore'):
                    means = H_s / H_c
                means[np.isnan(means)] = 0.0
                s = float(np.sum(H_c * means**2))
                feat_agg[i] += s
                feat_agg[j] += s
            top_feats = list(np.argsort(feat_agg)[::-1][:20])
        else:
            top_feats = list(range(d))  # exhaustive

        # ── final screening ────────────────────────────────────────────────
        candidates = []
        for combo in itertools.combinations(top_feats, degree):
            X_sub = X[:, combo]
            H_s, _ = np.histogramdd(X_sub, bins=10, weights=residuals)
            H_c, _ = np.histogramdd(X_sub, bins=10)
            with np.errstate(divide='ignore', invalid='ignore'):
                means = H_s / H_c
            means[np.isnan(means)] = 0.0
            candidates.append((float(np.sum(H_c * means**2)), combo))
        candidates.sort(key=lambda x: x[0], reverse=True)
        return [c[1] for c in candidates[:self.interactions]]

    # ── Predict a single term ────────────────────────────────────────────────
    def _predict_term(self, models, X_subset, is_binned=False):
        if is_binned:
            return np.sum(models, axis=0)[X_subset.ravel()]
        pred = np.zeros(X_subset.shape[0])
        for tree in models:
            pred += tree.predict(X_subset) * self.learning_rate
        return pred

    # ── BIC backward pruning (UPDATED: k_params cap for main effects) ────────
    def _backward_prune(self, X_binned, X, y_scaled):
        """
        One-at-a-time backward pruning via BIC

        Decision rule: remove term i if
            SSE_without_i - SSE_full < penalty_scale * log(n) * k_i
        i.e. the gain from keeping term i does not justify its complexity.
        """
        if self.penalty_scale <= 0.0:
            return

        n = len(y_scaled)
        pred = np.full(n, self.global_mean_)
        term_preds = []
        for i, feats in enumerate(self.term_features_):
            if len(feats) == 1:
                p = self._predict_term(self.additive_terms_[i],
                                       X_binned[:, feats], is_binned=True)
            else:
                p = self._predict_term(self.additive_terms_[i],
                                       X[:, feats], is_binned=False)
            term_preds.append(p)
            pred += p

        full_sse = float(np.sum((y_scaled - pred) ** 2))
        to_remove = []

        for i, feats in enumerate(self.term_features_):

            k = 1   # uniform: each term = 1 complexity unit

            if self.selection_criterion == "bic":
                penalty = self.penalty_scale * np.log(n) * k
            elif self.selection_criterion == "aic":
                penalty = self.penalty_scale * 2.0 * k
            else:
                continue

            sse_without = float(np.sum((y_scaled - (pred - term_preds[i])) ** 2))
            if (sse_without - full_sse) < penalty:
                to_remove.append(i)

        for i in sorted(to_remove, reverse=True):
            self.term_features_.pop(i)
            self.term_names_.pop(i)
            self.additive_terms_.pop(i)

    # ── fit ──────────────────────────────────────────────────────────────────
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.y_mean_ = float(np.mean(y))
        self.y_std_  = float(np.std(y)) if np.std(y) > 1e-9 else 1.0
        y_scaled     = (y - self.y_mean_) / self.y_std_

        # Discretise (use shared bins if provided by bagging wrapper)
        if not self.bins:
            X_binned, self.bins = self._discretize(X)
        else:
            X_binned = self._apply_bins(X)

        self.global_mean_ = float(np.mean(y_scaled))
        current_pred = np.full(n_samples, self.global_mean_)
        residuals    = y_scaled - current_pred

        # ── PHASE 1: discovery warmup (cyclic, T_warm rounds) ────────────────
        T_warm = max(50, self.n_estimators // 10)
        for _ in range(T_warm):
            for f_idx in range(n_features):
                n_bins = len(self.bins[f_idx]) - 1
                upd = self._fast_stump_learner(X_binned[:, f_idx], residuals, n_bins)
                upd *= self.learning_rate
                residuals -= upd[X_binned[:, f_idx]]

        # ── PHASE 1: interaction detection (on main-effect residuals) ────
        detected = []
        if self.interactions > 0:
            for deg in range(2, self.max_interaction_degree + 1):
                detected.extend(self._find_interactions(X, residuals, degree=deg))
        
        # ── OOB early stopping setup ────────────────────────────────────────
        oob_X_raw = getattr(self, '_oob_X_', None)
        if (oob_X_raw is not None and len(oob_X_raw) > 0
                and self.early_stopping_rounds > 0):
            oob_X_binned = self._apply_bins(oob_X_raw)
            oob_y_scaled = (self._oob_y_raw_ - self.y_mean_) / self.y_std_
            oob_pred = np.full(len(oob_y_scaled), self.global_mean_)
            best_oob_sse = np.inf
            es_patience  = 0
            es_check_every = 1
            use_es = True
        else:
            use_es = False

        # ── PHASE 2 ───────────────────────────────────────────────────────────
        if self.enable_joint_refit:
            # ── Joint cyclic refit (our approach) ────────────────────────────
            # All terms (mains + interactions) are reset and jointly refitted
            # from scratch for n_estimators epochs. Main effects are NOT frozen;
            # they re-adjust their signal sharing with interaction terms.
            self.term_features_ = [[i] for i in range(n_features)]
            self.term_names_    = [f"Feature_{i}" for i in range(n_features)]
            self.additive_terms_= [[] for _ in range(n_features)]

            for tup in detected:
                self.term_features_.append(list(tup))
                self.term_names_.append("Interaction_" + "_".join(map(str, tup)))
                self.additive_terms_.append([])

            current_pred = np.full(n_samples, self.global_mean_)
            residuals    = y_scaled - current_pred

            for epoch in range(self.n_estimators):
                for t_idx, feats in enumerate(self.term_features_):
                    if len(feats) == 1:
                        f_idx  = feats[0]
                        n_bins = len(self.bins[f_idx]) - 1
                        upd    = self._fast_stump_learner(X_binned[:, f_idx], residuals, n_bins)
                        upd   *= self.learning_rate
                        self.additive_terms_[t_idx].append(upd)
                        residuals -= upd[X_binned[:, f_idx]]
                        if use_es:
                            oob_pred += upd[oob_X_binned[:, f_idx]]
                    else:
                        tree = DecisionTreeRegressor(
                            max_depth       = len(feats),
                            max_leaf_nodes  = 2 ** len(feats),
                            min_samples_leaf= 20)
                        tree.fit(X[:, feats], residuals)
                        self.additive_terms_[t_idx].append(tree)
                        residuals -= tree.predict(X[:, feats]) * self.learning_rate
                        if use_es:
                            oob_pred += tree.predict(oob_X_raw[:, feats]) * self.learning_rate

                # ── OOB early stopping check ─────────────────────────
                if use_es and (epoch + 1) % es_check_every == 0:
                    oob_sse = float(np.sum((oob_y_scaled - oob_pred) ** 2))
                    if oob_sse < best_oob_sse * (1 - 1e-5):
                        best_oob_sse = oob_sse
                        es_patience  = 0
                    else:
                        es_patience += 1
                    if es_patience >= self.early_stopping_rounds:
                        self.best_iteration_ = epoch + 1
                        break

            if self.best_iteration_ is None:
                self.best_iteration_ = self.n_estimators

        else:
            # ── Two-stage fitting (interpretML-style) ─────────────────────────
            # Stage 1: mains fully fitted for n_estimators epochs, then frozen.
            # Stage 2: interactions fitted on frozen-main residuals.
            # Mirrors Lou et al. (2013) / interpretML GA2M approach.
            self.term_features_ = [[i] for i in range(n_features)]
            self.term_names_    = [f"Feature_{i}" for i in range(n_features)]
            self.additive_terms_= [[] for _ in range(n_features)]

            # Stage 1: full main-effect boosting
            current_pred = np.full(n_samples, self.global_mean_)
            residuals    = y_scaled - current_pred
            for epoch in range(self.n_estimators):
                for t_idx, feats in enumerate(self.term_features_):
                    f_idx  = feats[0]
                    n_bins = len(self.bins[f_idx]) - 1
                    upd    = self._fast_stump_learner(X_binned[:, f_idx], residuals, n_bins)
                    upd   *= self.learning_rate
                    self.additive_terms_[t_idx].append(upd)
                    residuals -= upd[X_binned[:, f_idx]]

            # Stage 2: interaction terms on frozen-main residuals
            for tup in detected:
                self.term_features_.append(list(tup))
                self.term_names_.append("Interaction_" + "_".join(map(str, tup)))
                self.additive_terms_.append([])

            interaction_indices = list(range(n_features, len(self.term_features_)))
            for epoch in range(self.n_estimators):
                for t_idx in interaction_indices:
                    feats = self.term_features_[t_idx]
                    tree = DecisionTreeRegressor(
                        max_depth       = len(feats),
                        max_leaf_nodes  = 2 ** len(feats),
                        min_samples_leaf= 20)
                    tree.fit(X[:, feats], residuals)
                    self.additive_terms_[t_idx].append(tree)
                    residuals -= tree.predict(X[:, feats]) * self.learning_rate


        # ── BIC backward pruning ─────────────────────────────────────────
        X_binned_full = self._apply_bins(X)
        self._backward_prune(X_binned_full, X, y_scaled)
        return self

    # ── predict ──────────────────────────────────────────────────────────────
    def predict(self, X):
        if not self.bins:
            raise ValueError("Model not fitted.")
        X_binned  = self._apply_bins(X)
        pred      = np.full(X.shape[0], self.global_mean_)
        for i, feats in enumerate(self.term_features_):
            if len(feats) == 1:
                pred += self._predict_term(self.additive_terms_[i],
                                           X_binned[:, feats], is_binned=True)
            else:
                pred += self._predict_term(self.additive_terms_[i],
                                           X[:, feats], is_binned=False)
        return pred * self.y_std_ + self.y_mean_


# ─────────────────────────────────────────────────────────────────────────────
class Sparse3WayEBMWithBagging(BaseEstimator, RegressorMixin):
    """
    Bagging wrapper around Sparse3WayEBM.
    Interaction terms are selected by majority vote across bags.

    Added attributes (reporting only, no algorithm change):
        avg_bag_term_count_      : mean number of terms per bag after BIC pruning.
        avg_bag_n_interactions_  : mean number of interaction terms per bag.
    These are the correct metrics for H3 (BIC reduces terms).
    """
    def __init__(self, outer_bags=5, n_jobs=1, max_interaction_degree=2, **ebm_kwargs):
        self.outer_bags             = outer_bags
        self.n_jobs                 = n_jobs
        self.max_interaction_degree = max_interaction_degree
        self.ebm_kwargs             = ebm_kwargs
        self.global_bins            = []
        self.models_                = []
        self.term_features_         = []
        self.term_names_            = []
        self.avg_bag_term_count_    = None   # NEW (reporting)
        self.avg_bag_n_interactions_= None   # NEW (reporting)

    def _discretize_global(self, X):
        n_bins   = self.ebm_kwargs.get("max_bins", 32)
        X_binned = np.zeros_like(X, dtype=int)
        self.global_bins = []
        for i in range(X.shape[1]):
            unique = np.unique(X[:, i])
            if len(unique) <= n_bins:
                edges = np.concatenate([[-np.inf], np.sort(unique)[1:], [np.inf]])
            else:
                edges = np.percentile(X[:, i], np.linspace(0, 100, n_bins + 1))
                edges[0], edges[-1] = -np.inf, np.inf
            self.global_bins.append(edges)
            X_binned[:, i] = np.clip(np.digitize(X[:, i], edges) - 1,
                                     0, len(edges) - 2)
        return X_binned

    def _fit_single_bag(self, X_full, y_full, bag_idx, seed):
        m = Sparse3WayEBM(max_interaction_degree=self.max_interaction_degree,
                             **self.ebm_kwargs)
        m.bins         = copy.deepcopy(self.global_bins)
        m.random_state = seed

        # OOB data for early stopping
        oob_mask = np.ones(len(y_full), dtype=bool)
        oob_mask[np.unique(bag_idx)] = False
        oob_idx = np.where(oob_mask)[0]
        if len(oob_idx) > 0:
            m._oob_X_      = X_full[oob_idx]
            m._oob_y_raw_  = y_full[oob_idx]
        else:
            m._oob_X_      = None
            m._oob_y_raw_  = None

        m.fit(X_full[bag_idx], y_full[bag_idx])
        return m

    def fit(self, X, y):
        self._discretize_global(X)
        self._X_full = X
        self._y_full = y
        seeds = np.random.randint(0, 10000, size=self.outer_bags)

        def _run(seed):
            np.random.seed(seed)
            idx = np.random.choice(len(y), len(y), replace=True)
            return self._fit_single_bag(X, y, idx, seed)

        self.models_ = Parallel(n_jobs=self.n_jobs)(delayed(_run)(s) for s in seeds)

        # ── Aggregate: mains always kept, interactions by vote ────────────
        n_features = X.shape[1]
        self.term_features_ = [[i] for i in range(n_features)]
        self.term_names_    = [f"Feature_{i}" for i in range(n_features)]

        all_tuples = []
        for m in self.models_:
            for f in m.term_features_:
                if len(f) > 1:
                    all_tuples.append(tuple(sorted(f)))

        target = self.ebm_kwargs.get("interactions", 0)
        if target > 0 and all_tuples:
            # Her degree için ayrı ayrı vote et
            for degree in range(2, self.max_interaction_degree + 1):
                degree_tuples = [t for t in all_tuples if len(t) == degree]
                if degree_tuples:
                    for tup, _ in Counter(degree_tuples).most_common(target):
                        self.term_features_.append(list(tup))
                        self.term_names_.append("Interaction_" + "_".join(map(str, tup)))

        # ── NEW: per-bag term counts (after BIC pruning) ──────────────────
        # These are the correct metrics for H3 — the wrapper's term_features_
        # always lists all mains + top-K interactions regardless of BIC.
        self.avg_bag_term_count_ = float(np.mean([
            len(m.term_features_) for m in self.models_
        ]))
        self.avg_bag_n_interactions_ = float(np.mean([
            sum(1 for f in m.term_features_ if len(f) > 1)
            for m in self.models_
        ]))
        return self

    def predict(self, X):
        return np.mean([m.predict(X) for m in self.models_], axis=0)

print("Core algorithm classes defined.")

## BLOCK 3 — Model Wrappers
Key addition: `supports_found_rate()` method.
- `True`  → ScratchEBMWrapper, VanillaEBMWrapper (term structure accessible)
- `False` → XGBWrapper, RFWrapper (black-box; found_rate reported as NaN = N/A)


In [ ]:
class ModelWrapper:
    def name(self):                raise NotImplementedError
    def fit(self, X, y):           raise NotImplementedError
    def predict(self, X):          raise NotImplementedError
    def term_names(self):          return []
    def term_features(self):       return []
    def get_selected_triplets(self): return []
    def avg_bag_term_count(self):  return np.nan
    def supports_found_rate(self): return False   # default: black-box


# ── Scratch EBM ──────────────────────────────────────────────────────────────
class ScratchEBMWrapper(ModelWrapper):
    def __init__(self, **params):
        self.params = params
        self.model  = Sparse3WayEBMWithBagging(**params)

    def name(self):
        deg = self.params.get("max_interaction_degree", 2)
        tag = "Scratch3Way" if deg == 3 else "Scratch2Way"
        ps  = float(self.params.get("penalty_scale", 0.0))
        if ps != 0.0:
            tag += f"_BIC{ps}"
        return tag

    def supports_found_rate(self): return True

    def fit(self, X, y):
        self.model = Sparse3WayEBMWithBagging(**self.params)   # fresh instance per fit
        self.model.fit(X, y)
        return self

    def predict(self, X): return self.model.predict(X)

    def term_names(self):
        for attr in ["term_names_", "global_term_names_"]:
            v = getattr(self.model, attr, None)
            if v is not None: return list(v)
        return []

    def term_features(self):
        for attr in ["term_features_", "global_term_features_"]:
            v = getattr(self.model, attr, None)
            if v is not None: return list(v)
        return []

    def get_selected_triplets(self):
        return [tuple(int(x) for x in f)
                for f in self.term_features() if len(f) == 3]

    def avg_bag_term_count(self):
        v = getattr(self.model, "avg_bag_term_count_", None)
        return float(v) if v is not None else np.nan


# ── Vanilla EBM ──────────────────────────────────────────────────────────────
class VanillaEBMWrapper(ModelWrapper):
    def __init__(self, **params):
        self.params = params
        self.model  = None

    def name(self): return "VanillaEBM"
    def supports_found_rate(self): return True    # pairwise only → always returns []

    def fit(self, X, y):
        if not HAS_INTERPRET:
            raise RuntimeError("interpretML not installed.")
        self.model = ExplainableBoostingRegressor(**self.params)
        self.model.fit(X, y)
        return self

    def predict(self, X): return self.model.predict(X)

    def term_names(self):
        v = getattr(self.model, "term_names_", None)
        return list(v) if v is not None else []

    def term_features(self):
        v = getattr(self.model, "term_features_", None)
        return list(v) if v is not None else []

    def get_selected_triplets(self): return []   # always empty: pairwise only

    def avg_bag_term_count(self):
        # VanillaEBM: just count the terms it selected
        return float(len(self.term_names()))


# ── XGBoost ──────────────────────────────────────────────────────────────────
class XGBWrapper(ModelWrapper):
    def __init__(self, **params):
        self.params = params
        self.model  = None

    def name(self): return "XGBoost"
    # supports_found_rate() = False (inherited)

    def fit(self, X, y):
        if not HAS_XGBOOST:
            raise RuntimeError("xgboost not installed.")
        self.model = XGBRegressor(**self.params)
        self.model.fit(X, y)
        return self

    def predict(self, X): return self.model.predict(X)


# ── Random Forest ─────────────────────────────────────────────────────────────
class RFWrapper(ModelWrapper):
    def __init__(self, **params):
        self.params = params
        self.model  = None

    def name(self): return "RandomForest"
    # supports_found_rate() = False (inherited)

    def fit(self, X, y):
        self.model = RandomForestRegressor(**self.params)
        self.model.fit(X, y)
        return self

    def predict(self, X): return self.model.predict(X)


print("Wrappers defined.")
print("supports_found_rate: Scratch=True, VanillaEBM=True, XGB=False, RF=False")


## BLOCK 4 — Dataset Suite

**Synthetic:** Friedman1, Ishigami, Hartmann6D (standard + modified with planted 3-way signal)

**Real (OpenML):** 7 datasets chosen for size/feature diversity.

| Dataset | n | p | Size | Features |
|---------|---|---|------|---------|
| red_wine | 1,599 | 11 | small | few |
| abalone | 4,177 | 7 | small | few |
| cpu_act | 8,192 | 21 | medium | medium |
| superconduct | 21,263 | 81 | large | many |
| diamonds | 53,940 | 6 | large | few |
| elevators | 16,599 | 18 | medium | medium |
| naval_propulsion_plant | 11,934 | 14 | medium | medium |

Sources: Fischer et al. (2023). OpenML-CTR23 (red_wine, abalone, cpu_act, superconduct, diamonds, naval_propulsion_plant). elevators from OpenML.

In [98]:
# ── Synthetic dataset names ───────────────────────────────────────────────────
DATASETS_SYNTH_STANDARD  = ["Friedman1", "Ishigami", "Hartmann6D"]
DATASETS_SYNTH_MOD       = ["Friedman1_Mod", "Ishigami_Mod", "Hartmann6D_Mod"]
DATASETS_SYNTH_MOD_NOISE = [d.replace("_Mod", "_ModNoise10") for d in DATASETS_SYNTH_MOD]

# ── Real dataset names ────────────────────────────────────────────────────────
# Source: Fischer et al. (2023). OpenML-CTR23.
# OpenML study id=353. https://openreview.net/forum?id=HebAOoMm94
DATASETS_REAL = ["red_wine", "abalone", "cpu_act", "superconduct", "diamonds",
                  "elevators", "naval_propulsion_plant"]

OPENML_IDS = {
    "red_wine":  40691,
    "abalone":       183,
    "cpu_act":       197,
    "superconduct":  43174,
    "diamonds":      42225,
    "elevators":     216,
    "naval_propulsion_plant":     44969,
}


In [99]:
# ── Synthetic helper functions ─────────────────────────────────────────────────
def _ishigami_base(X):
    return np.sin(X[:,0]) + 7*np.sin(X[:,1])**2 + 0.1*(X[:,2]**4)*np.sin(X[:,0])

def _hartmann6_base(X):
    alpha = np.array([1.0, 1.2, 3.0, 3.2])
    A = np.array([[10,3,17,3.5,1.7,8],[0.05,10,17,0.1,8,14],
                  [3,3.5,1.7,10,17,8],[17,8,0.05,10,0.1,14]])
    P = 1e-4*np.array([[1312,1696,5569,124,8283,5886],[2329,4135,8307,3736,1004,9991],
                        [2348,1451,3522,2883,3047,6650],[4047,8828,8732,5743,1091,381]])
    y = np.zeros(len(X))
    for i in range(4):
        y -= alpha[i] * np.exp(-np.sum(A[i] * (X - P[i])**2, axis=1))
    return y

def get_synth_dataset(name, n_samples=2000, seed=42):
    rng  = np.random.RandomState(seed)
    meta = {"name": name, "n_samples": n_samples, "seed": seed, "truth_triplet": None}

    if name == "Friedman1":
        X, y = make_friedman1(n_samples=n_samples, n_features=10, noise=1.0, random_state=seed)
        return X, y, meta

    if name == "Friedman1_Mod":
        X, y = make_friedman1(n_samples=n_samples, n_features=10, noise=0.0, random_state=seed)
        y   += 25.0 * np.sin(np.pi * X[:,0] * X[:,1] * X[:,2])
        meta["truth_triplet"] = (0, 1, 2)
        return X, y, meta

    if name == "Ishigami":
        X = rng.uniform(-np.pi, np.pi, (n_samples, 3))
        y = _ishigami_base(X) + rng.normal(0, 0.1, n_samples)
        return X, y, meta

    if name == "Ishigami_Mod":
        X = rng.uniform(-np.pi, np.pi, (n_samples, 3))
        y = _ishigami_base(X) + 15.0*np.sin(X[:,0])*np.sin(X[:,1])*np.sin(X[:,2])
        y += rng.normal(0, 0.1, n_samples)
        meta["truth_triplet"] = (0, 1, 2)
        return X, y, meta

    if name == "Hartmann6D":
        X = rng.uniform(0, 1, (n_samples, 6))
        y = _hartmann6_base(X) + rng.normal(0, 0.01, n_samples)
        return X, y, meta

    if name == "Hartmann6D_Mod":
        X = rng.uniform(0, 1, (n_samples, 6))
        y = _hartmann6_base(X) + 20.0*(X[:,0]*X[:,3]*X[:,5])
        y += rng.normal(0, 0.01, n_samples)
        meta["truth_triplet"] = (0, 3, 5)
        return X, y, meta

    raise ValueError(f"Unknown synthetic dataset: {name}")


In [100]:
# ── Noise-augmented variant ────────────────────────────────────────────────────
_NOISE_K      = 10
_NOISE_SUFFIX = "_ModNoise10"

def get_synth_noise_dataset(name, n_samples=2000, seed=42):
    """Load a *_ModNoise10 dataset: base _Mod + 10 appended uniform noise features."""
    base_name = name.replace(_NOISE_SUFFIX, "_Mod")
    X, y, meta = get_synth_dataset(base_name, n_samples=n_samples, seed=seed)
    rng  = np.random.RandomState(seed + 99999)
    Z    = rng.uniform(0, 1, (X.shape[0], _NOISE_K))
    Xn   = np.hstack([X, Z])
    meta2 = dict(meta, base_dataset=base_name,
                 noise_k=_NOISE_K, p_original=X.shape[1], p=Xn.shape[1])
    return Xn, y, meta2


In [101]:
# ── Real dataset loader ────────────────────────────────────────────────────────
_real_cache = {}   # cache so each dataset is fetched only once

def get_real_dataset(name):
    """
    Fetch a real dataset from OpenML (CTR23 suite).
    Results are cached in memory to avoid repeated downloads.
    """
    if name in _real_cache:
        return _real_cache[name]

    oid = OPENML_IDS[name]
    print(f"  Fetching {name} (OpenML id={oid}) ...", flush=True)
    ds  = fetch_openml(data_id=oid, as_frame=True, parser="auto")

    X = ds.data.select_dtypes(include=[np.number]).values.astype(float)
    y_raw = ds.target
    if hasattr(y_raw, "values"):
        y_raw = y_raw.values
    y = y_raw.astype(float)

    # drop rows with NaN
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X, y = X[mask], y[mask]

    feat_names = list(ds.data.select_dtypes(include=[np.number]).columns)
    meta = {"name": name, "n": X.shape[0], "p": X.shape[1],
            "feature_names": feat_names, "truth_triplet": None}
    _real_cache[name] = (X, y, meta)
    print(f"    -> n={X.shape[0]}, p={X.shape[1]}")
    return X, y, meta


In [ ]:
# ── Unified data getter ────────────────────────────────────────────────────────
def get_data(name):
    if name in DATASETS_REAL:
        return get_real_dataset(name)
    if name.endswith(_NOISE_SUFFIX):
        return get_synth_noise_dataset(name, n_samples=SYNTH_N, seed=42)
    return get_synth_dataset(name, n_samples=SYNTH_N, seed=42)

# ── Dataset summary table ─────────────────────────────────────────────────────
rows = []
for ds in DATASETS_SYNTH_STANDARD + DATASETS_SYNTH_MOD:
    X, y, meta = get_synth_dataset(ds, n_samples=SYNTH_N, seed=42)
    rows.append({"dataset": ds, "type": "synthetic", "n": X.shape[0],
                 "p": X.shape[1], "truth_triplet": str(meta["truth_triplet"])})
print("Synthetic datasets loaded.")
print("\nSkipping real dataset fetch in this cell (will be fetched on demand).")
print("Real datasets:", DATASETS_REAL)

df_ds_synth = pd.DataFrame(rows)
display(df_ds_synth)
export_table(df_ds_synth, "DatasetSummary_Synthetic")


## BLOCK 5 — Splits
Single source of truth for all train/test indices. Run once, reload on subsequent runs.

In [103]:
def get_split_indices(n, seed):
    rng = np.random.RandomState(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_test = int(n * TEST_SIZE)
    return idx[n_test:], idx[:n_test]    # (train_idx, test_idx)

def load_split(dataset, seed):
    """Load pre-saved split indices. Noise datasets reuse base _Mod splits."""
    base = dataset.replace(_NOISE_SUFFIX, "_Mod") if dataset.endswith(_NOISE_SUFFIX) else dataset
    tr = np.load(os.path.join(DIRS["splits"], f"{base}_seed{seed}_train.npy"))
    te = np.load(os.path.join(DIRS["splits"], f"{base}_seed{seed}_test.npy"))
    return tr, te


In [ ]:
ALL_SPLIT_DATASETS = DATASETS_SYNTH_STANDARD + DATASETS_SYNTH_MOD + DATASETS_REAL

manifest = []
for ds in ALL_SPLIT_DATASETS:
    X, y, _ = get_data(ds)
    for seed in GLOBAL_SEEDS:
        tr, te = get_split_indices(len(y), seed)
        np.save(os.path.join(DIRS["splits"], f"{ds}_seed{seed}_train.npy"), tr)
        np.save(os.path.join(DIRS["splits"], f"{ds}_seed{seed}_test.npy"),  te)
        manifest.append({"dataset": ds, "seed": seed,
                         "n_train": len(tr), "n_test": len(te)})

save_json(manifest, os.path.join(DIRS["splits"], "SplitManifest.json"))
df_manifest = pd.DataFrame(manifest)
print(df_manifest.groupby("dataset")[["n_train","n_test"]].first().to_string())
print(f"\nSplits saved for {len(ALL_SPLIT_DATASETS)} datasets x {len(GLOBAL_SEEDS)} seeds.")


## BLOCK 6 — Hyperparameter Tuning (Per-Dataset)

Search spaces adapted from:
> Bischl et al. (2023). *Hyperparameter Optimization: Foundations, Algorithms,
> Best Practices and Open Challenges.* WIREs Data Mining.
> https://github.com/mlr-org/mlr3tuningspaces

**Fair comparison rule:** 3-Way EBM models and VanillaEBM share the same
`EBM_BEST_PARAMS[dataset]` (shared knobs). XGBoost and RF are tuned
independently in their own search spaces.

**Speed:** `TUNING_N_SPLITS=3` inner CV folds per config (set in Block 0).
Results are cached to JSON — re-running Block 6 skips tuning if cache exists.


In [ ]:
# ── Search spaces (Bischl et al. 2023) ───────────────────────────────────────
# Grids are deliberately compact for practical runtime.
# Ranges are informed by Bischl et al. (2023) and mlr3tuningspaces defaults.
# Ref: https://github.com/mlr-org/mlr3tuningspaces

# EBM / Scratch — shared knobs (fair comparison contract)
# 8 configs × ~10s × 3 folds × 8 datasets ≈ 32 min
EBM_GRID = {
    "n_estimators":  [2000, 5000],   # Bischl: large n_rounds beneficial for EBM
    "learning_rate": [0.01, 0.05, 0.1],   # Bischl: log-scale 0.005-0.3
    "max_bins":      [32],           # fixed: 32 wins in pre-experiments; keeps k_params low
    "interactions":  [3,5],         # controls candidate pool for screening select at least 5 to give possiblity to find 3 way interactions
    "outer_bags":    [5],            # fixed for stability
}

# XGBoost — Bischl et al. (2023) Table 1 recommended space (compressed)
# 32 configs × ~1.5s × 3 folds × 8 datasets ≈ 12 min
XGB_GRID = {
    "n_estimators":      [300, 1000],   # Bischl: 100-2000 (log-scale)
    "learning_rate":     [0.05, 0.1],   # Bischl: 0.005-0.3 (log-scale)
    "max_depth":         [3, 6],        # Bischl: 1-15
    "subsample":         [0.75, 1.0],   # Bischl: 0.1-1.0
    "colsample_bytree":  [0.75, 1.0],   # Bischl: 0.1-1.0
    "reg_lambda":        [1.0],         # fixed to default (Bischl: 0.001-1000 log)
    "min_child_weight":  [1],           # fixed to default (Bischl: 1-100 log)
}

# Random Forest — Bischl et al. (2023) ranger-based space (compressed)
# 8 configs × ~2s × 3 folds × 8 datasets ≈ 6 min
RF_GRID = {
    "n_estimators":     [300],           # RF not sensitive; Bischl: 1-2000
    "max_features":     ["sqrt", 0.33],  # Bischl: mtry.ratio 0.1-0.9
    "min_samples_leaf": [1, 5],          # Bischl: 1-100 (log-scale)
    "max_depth":        [None, 10],      # Bischl: unbounded or moderate cap
}

from sklearn.model_selection import ParameterGrid
print(f"EBM grid: {len(list(ParameterGrid(EBM_GRID)))} configs")
print(f"XGB grid: {len(list(ParameterGrid(XGB_GRID)))} configs")
print(f"RF  grid: {len(list(ParameterGrid(RF_GRID)))} configs")
print(f"Estimated total tuning time: ~20-30 min (8 datasets x 3 folds)")

TUNING_CACHE_PATH = os.path.join(DIRS["hparam"], "PerDataset_BestParams.json")

In [106]:
# ── Tuning functions ──────────────────────────────────────────────────────────
def _cv_rmse(model_fn, X, y, n_splits, seed_offset=0):
    """Simple repeated random holdout CV. Returns mean RMSE."""
    scores = []
    for k in range(n_splits):
        Xtr, Xval, ytr, yval = train_test_split(
            X, y, test_size=0.2, random_state=seed_offset + k)
        m = model_fn()
        m.fit(Xtr, ytr)
        scores.append(rmse(yval, m.predict(Xval)))
    return float(np.mean(scores))

def tune_ebm(dataset, n_splits=TUNING_N_SPLITS):
    X, y, _ = get_data(dataset)
    configs  = list(ParameterGrid(EBM_GRID))
    best_rmse, best_cfg = np.inf, None
    for i, cfg in enumerate(configs, 1):
        def _make(c=cfg):
            return Sparse3WayEBMWithBagging(
                max_interaction_degree=2, penalty_scale=0.0,
                enable_joint_refit=True,n_jobs=N_JOBS, **c)
        s = _cv_rmse(_make, X, y, n_splits)
        print(f"    EBM [{i}/{len(configs)}] rmse={s:.4f}  {cfg}", flush=True)
        if s < best_rmse:
            best_rmse, best_cfg = s, cfg
    print(f"  -> EBM best: rmse={best_rmse:.4f}  {best_cfg}")
    return best_cfg, best_rmse

def tune_xgb(dataset, n_splits=TUNING_N_SPLITS):
    if not HAS_XGBOOST:
        default = {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 6,
                   "subsample": 1.0, "colsample_bytree": 1.0,
                   "reg_lambda": 1.0, "min_child_weight": 1}
        print("  XGBoost not installed, using default params.")
        return default, np.nan
    X, y, _ = get_data(dataset)
    configs  = list(ParameterGrid(XGB_GRID))
    best_rmse, best_cfg = np.inf, None
    for i, cfg in enumerate(configs, 1):
        def _make(c=cfg):
            return XGBRegressor(objective="reg:squarederror",
                                tree_method="hist", n_jobs=N_JOBS, **c)
        s = _cv_rmse(_make, X, y, n_splits)
        print(f"    XGB [{i}/{len(configs)}] rmse={s:.4f}  {cfg}", flush=True)
        if s < best_rmse:
            best_rmse, best_cfg = s, cfg
    print(f"  -> XGB best: rmse={best_rmse:.4f}  {best_cfg}")
    return best_cfg, best_rmse

def tune_rf(dataset, n_splits=TUNING_N_SPLITS):
    X, y, _ = get_data(dataset)
    configs  = list(ParameterGrid(RF_GRID))
    best_rmse, best_cfg = np.inf, None
    for i, cfg in enumerate(configs, 1):
        def _make(c=cfg):
            return RandomForestRegressor(random_state=42, n_jobs=N_JOBS, **c)
        s = _cv_rmse(_make, X, y, n_splits)
        print(f"    RF  [{i}/{len(configs)}] rmse={s:.4f}  {cfg}", flush=True)
        if s < best_rmse:
            best_rmse, best_cfg = s, cfg
    print(f"  -> RF  best: rmse={best_rmse:.4f}  {best_cfg}")
    return best_cfg, best_rmse

In [ ]:
# ── Run tuning (or load from cache) ──────────────────────────────────────────
TUNE_DATASETS = DATASETS_SYNTH_MOD + DATASETS_REAL
if os.path.exists(TUNING_CACHE_PATH):
    print(f"Cache found: {TUNING_CACHE_PATH}")
    print("Loading tuned params from cache (delete file to re-run tuning).")
    with open(TUNING_CACHE_PATH) as f:
        _cache = json.load(f)
    EBM_BEST = _cache["EBM"]
    XGB_BEST = _cache["XGB"]
    RF_BEST  = _cache["RF"]
    print("Loaded:", list(EBM_BEST.keys()))
else:
    print(f"No cache found. Starting per-dataset tuning ({len(TUNE_DATASETS)} datasets).")
    print(f"TUNING_N_SPLITS={TUNING_N_SPLITS}\n")
    EBM_BEST, XGB_BEST, RF_BEST = {}, {}, {}

    for ds_i, ds in enumerate(TUNE_DATASETS, 1):
        print(f"{'='*60}")
        print(f"[{ds_i}/{len(TUNE_DATASETS)}] Tuning dataset: {ds}")
        print(f"{'='*60}")

        print("  -- EBM --")
        cfg_e, r_e = tune_ebm(ds)
        EBM_BEST[ds] = cfg_e

        print("  -- XGBoost --")
        cfg_x, r_x = tune_xgb(ds)
        XGB_BEST[ds] = cfg_x

        print("  -- RandomForest --")
        cfg_r, r_r = tune_rf(ds)
        RF_BEST[ds] = cfg_r

        # Her dataset sonrası kaydet — Colab disconnect'e karşı güvenlik
        save_json({"EBM": EBM_BEST, "XGB": XGB_BEST, "RF": RF_BEST},
                  TUNING_CACHE_PATH)
        print(f"  [saved to cache after {ds}]\n")

    print("Tuning complete.")

# ── Propagate params to standard / noise datasets ────────────────────────────
for std, mod in zip(DATASETS_SYNTH_STANDARD, DATASETS_SYNTH_MOD):
    EBM_BEST[std] = EBM_BEST[mod]
    XGB_BEST[std] = XGB_BEST[mod]
    RF_BEST[std]  = RF_BEST[mod]

for ds_noise in DATASETS_SYNTH_MOD_NOISE:
    base = ds_noise.replace(_NOISE_SUFFIX, "_Mod")
    EBM_BEST[ds_noise] = EBM_BEST[base]
    XGB_BEST[ds_noise] = XGB_BEST[base]
    RF_BEST[ds_noise]  = RF_BEST[base]

print("\nAll datasets have tuned params. Ready for Block 7.")

In [ ]:
# ── BLOCK 6b: BIC Scale Tuning for Real Datasets ─────────────────────────────
# Strategy: use best EBM params (already tuned, penalty_scale=0),
# sweep only penalty_scale with 1-fold CV. Fast: 5 values × 7 datasets.
#
# Results are merged into the existing cache JSON under key "BIC_SCALE".
# If cache already has "BIC_SCALE", loads from cache (skip re-run).

BIC_SCALE_CACHE_KEY = "BIC_SCALE"
BIC_SCALE_CANDIDATES = BIC_SCALES #[0.0, 0.1, 0.25, 0.5, 0.75, 1]
BIC_TUNING_N_SPLITS  = 3   

if BIC_SCALE_CACHE_KEY in json.load(open(TUNING_CACHE_PATH)) if os.path.exists(TUNING_CACHE_PATH) else {}:
    print("BIC_SCALE cache found — loading.")
    with open(TUNING_CACHE_PATH) as f:
        BIC_SCALE_BEST = json.load(f)[BIC_SCALE_CACHE_KEY]
    print("Loaded BIC_SCALE_BEST:", BIC_SCALE_BEST)
else:
    print("[BLOCK 6b] BIC scale tuning — real datasets only.")
    print(f"Candidates: {BIC_SCALE_CANDIDATES}  |  splits: {BIC_TUNING_N_SPLITS}\n")
    BIC_SCALE_BEST = {}

    for ds_i, ds in enumerate(DATASETS_REAL, 1):
        if ds not in EBM_BEST:
            print(f"  [{ds}] skipped — no EBM params.")
            continue

        print(f"[{ds_i}/{len(DATASETS_REAL)}] {ds}")
        X, y, _ = get_data(ds)
        ebm_p = dict(EBM_BEST[ds])

        best_rmse, best_ps = np.inf, 0.02
        for ps in BIC_SCALE_CANDIDATES:
            def _make_bic(ps=ps):
                return Sparse3WayEBMWithBagging(
                    max_interaction_degree=3,
                    penalty_scale=ps,
                    enable_joint_refit=True,
                    n_jobs=N_JOBS,
                    **ebm_p
                )
            s = _cv_rmse(_make_bic, X, y, n_splits=BIC_TUNING_N_SPLITS)
            print(f"    ps={ps:.3f}  rmse={s:.4f}", flush=True)
            if s < best_rmse:
                best_rmse, best_ps = s, ps

        BIC_SCALE_BEST[ds] = best_ps
        print(f"  → best ps={best_ps}  rmse={best_rmse:.4f}\n")

    # Merge into existing cache JSON
    if os.path.exists(TUNING_CACHE_PATH):
        with open(TUNING_CACHE_PATH) as f:
            _cache = json.load(f)
    else:
        _cache = {"EBM": EBM_BEST, "XGB": XGB_BEST, "RF": RF_BEST}

    _cache[BIC_SCALE_CACHE_KEY] = BIC_SCALE_BEST
    save_json(_cache, TUNING_CACHE_PATH)
    print("BIC_SCALE_BEST saved to cache.")

print("\nBIC_SCALE_BEST:", BIC_SCALE_BEST)

In [ ]:
# ── Fairness contract: EBM shared knobs == VanillaEBM knobs ──────────────────
# This cell documents that Scratch and VanillaEBM receive identical core params.
sample_ds = DATASETS_SYNTH_MOD[0]
ebm_p = EBM_BEST[sample_ds]

CONTRACT_KEYS = ["n_estimators", "learning_rate", "max_bins", "interactions", "outer_bags"]
vanilla_p = {
    "max_rounds":    ebm_p["n_estimators"],
    "learning_rate": ebm_p["learning_rate"],
    "max_bins":      ebm_p["max_bins"],
    "interactions":  ebm_p["interactions"],
    "outer_bags":    ebm_p["outer_bags"],
}
rows = [{"param": k, "Scratch_value": ebm_p.get(k), "VanillaEBM_value": vanilla_p.get(k, ebm_p.get(k))}
        for k in CONTRACT_KEYS]
df_contract = pd.DataFrame(rows)
display(df_contract)
export_table(df_contract, "FairnessContract")


## BLOCK 7 — Build Models
`build_models_for_dataset()` uses the per-dataset tuned params from Block 6.

In [ ]:
def _vanilla_params(ebm_p):
    """Map Scratch EBM params to VanillaEBM param names."""
    return dict(
        max_rounds    = ebm_p["n_estimators"],
        learning_rate = ebm_p["learning_rate"],
        max_bins      = ebm_p["max_bins"],
        interactions  = ebm_p["interactions"],
        outer_bags    = ebm_p["outer_bags"],
        random_state  = 42,
        inner_bags    = 0,
        n_jobs        = N_JOBS,
    )

def build_models_for_dataset(dataset, include_scratch2=None, include_bic=True, bic_scale=None,
                              early_stopping_rounds=200):
    """
    Build the model list for a given dataset using per-dataset tuned params.

    include_scratch2 : bool or None
        None → auto (True for synthetic, False for real)
    bic_scale : float or None
        None → use BIC_SCALE_BEST[dataset] if available, else 0.5 fallback.
        Explicit float → override.
    early_stopping_rounds : int
        0 → disabled (default). >0 → stop boosting early if val loss does not improve.
    """
    ebm_p = dict(EBM_BEST[dataset])
    xgb_p = dict(XGB_BEST[dataset])
    rf_p  = dict(RF_BEST[dataset])

    if include_scratch2 is None:
        include_scratch2 = (dataset not in DATASETS_REAL)

    # BIC scale: per-dataset tuned > explicit arg > global fallback
    if bic_scale is None:
        bic_scale = BIC_SCALE_BEST.get(dataset, 0.5)

    models = []

    if include_scratch2:
        models.append(ScratchEBMWrapper(
            max_interaction_degree=2, penalty_scale=0.0,
            enable_joint_refit=True, n_jobs=N_JOBS,
            early_stopping_rounds=early_stopping_rounds, **ebm_p))

    models.append(ScratchEBMWrapper(
        max_interaction_degree=3, penalty_scale=0.0,
        enable_joint_refit=True, n_jobs=N_JOBS,
        early_stopping_rounds=early_stopping_rounds, **ebm_p))

    if include_bic:
        models.append(ScratchEBMWrapper(
            max_interaction_degree=3, penalty_scale=bic_scale,
            enable_joint_refit=True, n_jobs=N_JOBS,
            early_stopping_rounds=early_stopping_rounds, **ebm_p))

    if HAS_INTERPRET:
        models.append(VanillaEBMWrapper(**_vanilla_params(ebm_p)))

    if HAS_XGBOOST:
        models.append(XGBWrapper(
            objective="reg:squarederror", tree_method="hist",
            n_jobs=N_JOBS, random_state=42, **xgb_p))

    models.append(RFWrapper(random_state=42, n_jobs=N_JOBS, **rf_p))

    return models

# Quick check
_demo = build_models_for_dataset("Friedman1_Mod")
print("Models for Friedman1_Mod:")
for m in _demo:
    print(f"  {m.name()} | supports_found_rate={m.supports_found_rate()}")


## BLOCK 8 — Benchmark Engine

`run_benchmark()` core changes:
- `found_truth_triplet = NaN` for models where `supports_found_rate() == False`
- `n_terms` = `avg_bag_term_count_` (post-BIC per-bag average, not wrapper count)
- `collect_terms=True` → returns a second DataFrame with per-term rows for qualitative analysis


In [ ]:
import time as _t

def run_benchmark(datasets, seeds, collect_terms=False, build_kwargs=None):

    """
    Run benchmark over datasets × seeds × models.

    Returns
    -------
    df       : per-run results (one row per dataset/seed/model)
    df_terms : per-term listing (only when collect_terms=True, else None)

    """
    if build_kwargs is None:
        build_kwargs = {}
    rows, term_rows = [], []

    n_datasets = len(datasets)
    n_seeds    = len(seeds)

    for ds_i, ds in enumerate(datasets, 1):
        X, y, meta = get_data(ds)
        truth  = meta.get("truth_triplet", None)
        models = build_models_for_dataset(ds, **build_kwargs)
        n_models    = len(models)

        print(f"\n{'='*60}")
        print(f"[{ds_i}/{n_datasets}] Dataset: {ds}  |  n={len(y)}  |  models={n_models}  |  seeds={n_seeds}")
        print(f"{'='*60}")

        for seed_i, seed in enumerate(seeds, 1):
            tr, te   = load_split(ds, seed)
            Xtr, ytr = X[tr], y[tr]
            Xte, yte = X[te], y[te]

            print(f"\n  [seed {seed_i}/{n_seeds}] seed={seed}  "
                  f"(train={len(tr)}, test={len(te)})")

            for m_i, m in enumerate(models, 1):
                mname = m.name()
                print(f"    [{m_i}/{n_models}] {mname:<30} fitting...",
                      end=" ", flush=True)

                t0 = _t.time()
                m.fit(Xtr, ytr)
                fit_t = _t.time() - t0
                # ── save each model ────────────────────────────────────────────
                model_path = os.path.join(DIRS["models"], f"{ds}_seed{seed}_{mname}.pkl")
                joblib.dump(m, model_path)

                pred      = m.predict(Xte)
                this_rmse = rmse(yte, pred)
                this_mae  = mae(yte, pred)
                this_r2   = r2(yte, pred)

                # ── found_rate ────────────────────────────────────────────
                if m.supports_found_rate() and truth is not None:
                    truth_t   = tuple(sorted(truth))
                    found     = any(tuple(sorted(t)) == truth_t
                                    for t in m.get_selected_triplets())
                    found_val = float(found)
                    found_str = f"found={bool(found)}"
                else:
                    found_val = np.nan
                    found_str = "found=N/A"

                # ── term count ────────────────────────────────────────────
                n_terms_val = m.avg_bag_term_count()
                terms_str   = (f"n_terms={n_terms_val:.1f}"
                               if not np.isnan(n_terms_val) else "n_terms=N/A")

                print(f"done ({fit_t:.1f}s)  "
                      f"rmse={this_rmse:.4f}  {found_str}  {terms_str}")

                rows.append({
                    "dataset": ds, "seed": seed, "model": mname,
                    "rmse": this_rmse, "mae": this_mae, "r2": this_r2,
                    "fit_time": fit_t,
                    "truth_triplet": str(truth),
                    "found_truth_triplet": found_val,
                    "n_terms": n_terms_val,
                })

                if collect_terms and m.supports_found_rate():
                    for tn in m.term_names():
                        term_rows.append({
                            "dataset": ds, "seed": seed, "model": mname, "term": tn
                        })

        print(f"\n  [{ds}] done. Total runs: {n_seeds * n_models}")

    print(f"\n{'='*60}")
    print(f"Benchmark complete. Total rows: {len(rows)}")
    print(f"{'='*60}")

    df       = pd.DataFrame(rows)
    df_terms = pd.DataFrame(term_rows) if collect_terms else None
    return df, df_terms

def summarize(df):
    """Aggregate per-run results to per-dataset/model summaries."""
    return (df.groupby(["dataset", "model"])
              .agg(
                  rmse_mean       = ("rmse",               "mean"),
                  rmse_std        = ("rmse",               "std"),
                  mae_mean        = ("mae",                "mean"),
                  r2_mean         = ("r2",                 "mean"),
                  fit_time_mean   = ("fit_time",           "mean"),
                  found_rate      = ("found_truth_triplet", lambda x: x.dropna().mean()),
                  n_terms_mean    = ("n_terms",            lambda x: x.dropna().mean()),
              )
              .reset_index())


def show_terms(df_terms, dataset, seed=0):
    """Pretty-print selected terms for a given dataset and seed."""
    sub = df_terms[(df_terms.dataset == dataset) & (df_terms.seed == seed)]
    for mname in sub.model.unique():
        terms = sub[sub.model == mname]["term"].tolist()
        print(f"\n  {dataset} | {mname} | seed={seed} ({len(terms)} terms):")
        for t in terms:
            print(f"    {t}")

print("Benchmark engine ready.")


## BLOCK 9 — Standard Synthetic Benchmark (H1 baseline)
No planted 3-way signal → found_rate = N/A for all models.

In [ ]:
print("[BLOCK 9] Standard Synthetic Benchmark")
df_std, _ = run_benchmark(DATASETS_SYNTH_STANDARD, GLOBAL_SEEDS, collect_terms=False,build_kwargs={"include_bic": False,"include_scratch2": False})
df_std_sum = summarize(df_std).sort_values(["dataset", "rmse_mean"])


display(df_std_sum)
export_table(df_std_sum, "Table_StandardSynthetic")

fig, axes = plt.subplots(1, len(DATASETS_SYNTH_STANDARD), figsize=(14, 4), sharey=False)
for ax, ds in zip(axes, DATASETS_SYNTH_STANDARD):
    sub = df_std_sum[df_std_sum.dataset == ds].sort_values("rmse_mean")
    ax.barh(sub["model"], sub["rmse_mean"], xerr=sub["rmse_std"], capsize=3)
    ax.set_title(ds); ax.set_xlabel("RMSE")
plt.suptitle("Standard Synthetic Benchmarks", fontsize=13)
export_fig(fig, "Fig_StandardSynthetic_RMSE")
plt.show()


## BLOCK 10 — Modified Synthetic Benchmark (H1 + H2)
Planted 3-way signal present. Tests:
- H1: 3-way EBM matches/beats Vanilla EBM on accuracy
- H2: found_rate — only Scratch and VanillaEBM get a value; XGB/RF → N/A


In [ ]:
print("[BLOCK 10] Modified Synthetic Benchmark")
df_mod, df_mod_terms = run_benchmark(DATASETS_SYNTH_MOD, GLOBAL_SEEDS, collect_terms=True,build_kwargs={"include_bic": True ,"bic_scale": 0.5, "include_scratch2": False, "early_stopping_rounds": 200})
export_table(df_mod, "Raw_ModifiedSynthetic_PerSeed")   # raw for reproducibility

df_mod_sum = summarize(df_mod).sort_values(["dataset", "rmse_mean"])
display(df_mod_sum)
export_table(df_mod_sum, "Table_ModifiedSynthetic")


In [ ]:
# ── Recovery table ────────────────────────────────────────────────────────────
rec_cols = ["dataset", "model", "found_rate", "n_terms_mean", "rmse_mean", "rmse_std"]
df_rec = df_mod_sum[rec_cols].copy()
print("\nRecovery summary (N/A = model cannot report found_rate):")
display(df_rec.sort_values(["dataset", "found_rate"], ascending=[True, False]))
export_table(df_rec, "Table_ModifiedSynthetic_Recovery")


In [ ]:
# ── RMSE plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(DATASETS_SYNTH_MOD), figsize=(14, 4), sharey=False)
for ax, ds in zip(axes, DATASETS_SYNTH_MOD):
    sub = df_mod_sum[df_mod_sum.dataset == ds].sort_values("rmse_mean")
    ax.barh(sub["model"], sub["rmse_mean"], xerr=sub["rmse_std"], capsize=3)
    ax.set_title(ds); ax.set_xlabel("RMSE")
plt.suptitle("Modified Synthetic Benchmarks (3-way signal planted)", fontsize=13)
export_fig(fig, "Fig_ModifiedSynthetic_RMSE")
plt.show()


## BLOCK 11 — BIC Sweep (H3: single-knob sparsity)
Sweep `penalty_scale` on noise-augmented datasets. Does BIC prune noise features without hurting accuracy?

In [ ]:
BIC_SCALES  = [0.1, 0.25, 0.5, 0.75, 1]   # BIC penalty scales to sweep over (0.0 is no BIC, already included)
print("[BLOCK 11] BIC Sweep on ModNoise10 datasets")

# Build BIC sweep model list for a dataset
def build_bic_sweep_models(dataset):
    ebm_p = dict(EBM_BEST[dataset])
    ebm_p["interactions"] = 5   # more candidates so BIC has something to prune
    models = []

    # Baseline: no BIC
    models.append(ScratchEBMWrapper(
        max_interaction_degree=3, penalty_scale=0.0,
        enable_joint_refit=True, n_jobs=N_JOBS, **ebm_p))

    for ps in BIC_SCALES:
        models.append(ScratchEBMWrapper(
            max_interaction_degree=3, penalty_scale=ps,
            enable_joint_refit=True, n_jobs=N_JOBS, **ebm_p))
    if HAS_INTERPRET:
        models.append(VanillaEBMWrapper(**_vanilla_params(ebm_p)))
    return models

rows_bic = []
for ds in DATASETS_SYNTH_MOD_NOISE:
    X, y, meta = get_data(ds)
    models_bic = build_bic_sweep_models(ds)
    for seed in GLOBAL_SEEDS:
        tr, te   = load_split(ds, seed)
        Xtr, ytr = X[tr], y[tr]
        Xte, yte = X[te], y[te]
        for m in models_bic:
            print(f"  [BIC] {ds} | seed={seed} | {m.name()} fitting...", end=" ", flush=True)
            t0 = _time.time()
            m.fit(Xtr, ytr)
            fit_t = _time.time() - t0
            pred = m.predict(Xte)
            r = rmse(yte, pred)
            nt = m.avg_bag_term_count()
            print(f"done ({fit_t:.0f}s) | RMSE={r:.4f} | n_terms={nt:.1f}", flush=True)
            rows_bic.append({
                "dataset": ds, "seed": seed, "model": m.name(),
                "rmse": r, "n_terms": nt,
                "found_truth_triplet": np.nan,
            })

df_bic     = pd.DataFrame(rows_bic)
df_bic_sum = (df_bic.groupby(["dataset", "model"])
              .agg(
                  rmse_mean = ("rmse", "mean"),
                  rmse_std  = ("rmse", "std"),
                  n_terms_mean = ("n_terms", lambda x: x.dropna().mean()),
              )
              .reset_index()
              .sort_values(["dataset", "rmse_mean"]))
display(df_bic_sum)
export_table(df_bic_sum, "Table_BIC_Sweep")

In [ ]:
# ── BIC tradeoff plot: RMSE vs term count ─────────────────────────────────────
fig, axes = plt.subplots(1, len(DATASETS_SYNTH_MOD_NOISE), figsize=(14, 4))
for ax, ds in zip(axes, DATASETS_SYNTH_MOD_NOISE):
    sub = df_bic_sum[(df_bic_sum.dataset == ds) &
                     (df_bic_sum.model.str.contains("Scratch3Way"))]

    def _ps(name):
        return float(name.split("_BIC")[-1]) if "_BIC" in name else 0.0

    sub = sub.copy()
    sub["ps"] = sub["model"].apply(_ps)
    sub = sub.sort_values("ps")

    ax.plot(sub["n_terms_mean"], sub["rmse_mean"], marker="o")
    for _, row in sub.iterrows():
        ax.annotate(f"ps={row['ps']}", (row["n_terms_mean"], row["rmse_mean"]),
                    fontsize=7, textcoords="offset points", xytext=(3,3))
    ax.set_title(ds.replace("_ModNoise10","\n+Noise10"))
    ax.set_xlabel("Avg terms (post-BIC)"); ax.set_ylabel("RMSE")

plt.suptitle("BIC Sweep: RMSE vs Term Count (H3)", fontsize=13)
export_fig(fig, "Fig_BIC_Sweep_RMSE_vs_Terms")
plt.show()


## BLOCK 12 — FAST-3D Recovery (H2: Recall@K)
Does the screening recover the planted truth triplet at various K values?

In [ ]:
print("[BLOCK 12] FAST-3D Recovery (Recall@K, Precision@K)")

K_LIST = [1, 3, 5, 10]
rows_rec = []

for ds in DATASETS_SYNTH_MOD:
    X, y, meta = get_data(ds)
    truth = meta["truth_triplet"]

    for seed in GLOBAL_SEEDS:
        tr, te   = load_split(ds, seed)
        ebm_p    = dict(EBM_BEST[ds])
        m = Sparse3WayEBMWithBagging(
            max_interaction_degree=3, penalty_scale=0.0,
            enable_joint_refit=True, **ebm_p)
        m.fit(X[tr], y[tr])

        # Collect ALL triplets from all bags (ordered by vote count)
        all_t = []
        for bag in m.models_:
            for f in bag.term_features_:
                if len(f) == 3:
                    all_t.append(tuple(sorted(f)))
        ordered = [t for t, _ in Counter(all_t).most_common()]

        truth_t = tuple(sorted(truth))
        for k in K_LIST:
            top_k = set(ordered[:k])
            recall = 1.0 if truth_t in top_k else 0.0
            prec   = (1.0 / len(top_k)) if (truth_t in top_k and top_k) else 0.0
            rows_rec.append({
                "dataset": ds, "seed": seed, "k": k,
                "recall": recall, "precision": prec,
                "n_discovered": len(set(all_t)),
            })

df_rec     = pd.DataFrame(rows_rec)
df_rec_sum = (df_rec.groupby(["dataset", "k"])
              .agg(recall_mean    = ("recall",      "mean"),
                   precision_mean = ("precision",   "mean"),
                   n_disc_mean    = ("n_discovered","mean"))
              .reset_index())
display(df_rec_sum)
export_table(df_rec_sum, "Table_FAST3D_RecallAtK")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for ds in DATASETS_SYNTH_MOD:
    sub = df_rec_sum[df_rec_sum.dataset == ds]
    ax.plot(sub["k"], sub["recall_mean"], marker="o", label=ds)
ax.set_xlabel("K"); ax.set_ylabel("Recall@K")
ax.set_title("FAST-3D Triplet Recovery (H2)")
ax.legend(); ax.set_ylim(-0.05, 1.1)
export_fig(fig, "Fig_FAST3D_RecallAtK")
plt.show()


## BLOCK 13 — Real Dataset Benchmark (H4: competitiveness)
Models: Scratch3Way, Scratch3Way_BIC, VanillaEBM, XGBoost, RF.
found_rate = N/A for all (no ground truth in real data).
Term listing: which 3-way interactions does the model select?


In [ ]:
print("[BLOCK 13] Real Dataset Benchmark")
# Run on tuned datasets only (skip datasets where EBM tuning failed, since EBM is the focus of the real benchmark).
TUNED_REAL = [ds for ds in DATASETS_REAL if ds in EBM_BEST]
print(f"Tuned datasets: {TUNED_REAL}")

df_real, df_real_terms = run_benchmark(
    TUNED_REAL, GLOBAL_SEEDS,
    collect_terms=True,
    build_kwargs={"include_bic": True, "include_scratch2": False}
)
export_table(df_real, "Raw_Real_PerSeed")

df_real_sum = summarize(df_real).sort_values(["dataset", "rmse_mean"])
display(df_real_sum)
export_table(df_real_sum, "Table_RealBenchmark")


In [ ]:
# ── Real benchmark RMSE plot ──────────────────────────────────────────────────
n_ds = len(DATASETS_REAL)
fig, axes = plt.subplots(1, n_ds, figsize=(4*n_ds, 4), sharey=False)
for ax, ds in zip(axes, DATASETS_REAL):
    sub = df_real_sum[df_real_sum.dataset == ds].sort_values("rmse_mean")
    ax.barh(sub["model"], sub["rmse_mean"], xerr=sub["rmse_std"], capsize=3)
    ax.set_title(ds); ax.set_xlabel("RMSE")
plt.suptitle("Real Dataset Benchmarks (H4)", fontsize=13)
export_fig(fig, "Fig_RealBenchmark_RMSE")
plt.show()


In [ ]:
# ── BLOCK 13b: Term Overlap Analysis (load pre-fitted models) ────────────────
import joblib
from itertools import combinations as _combs

OVERLAP_DATASETS = ["cpu_act", "elevators", "naval_propulsion_plant", "diamonds", "superconduct"]
OVERLAP_SEED     = 0
MODEL_DIR        = DIRS["models"]

print("=" * 65)
print("Term Overlap Analysis: Vanilla EBM pairs ∩ 3-way EBM triples")
print("=" * 65)

overlap_rows = []

for ds in OVERLAP_DATASETS:

    van_path = os.path.join(MODEL_DIR, f"{ds}_seed{OVERLAP_SEED}_VanillaEBM.pkl")
    scr_path = os.path.join(MODEL_DIR, f"{ds}_seed{OVERLAP_SEED}_Scratch3Way.pkl")

    van = joblib.load(van_path)
    scr = joblib.load(scr_path)

    vanilla_pairs = set(
        tuple(sorted(f)) for f in van.term_features() if len(f) == 2
    )
    scratch_pairs = set(
        tuple(sorted(f)) for f in scr.term_features() if len(f) == 2
    )
    scratch_triples = set(
        tuple(sorted(f)) for f in scr.term_features() if len(f) == 3
    )

    triple_overlap_info = []
    for triple in sorted(scratch_triples):
        sub_pairs   = [tuple(sorted(p)) for p in _combs(triple, 2)]
        van_overlap = [p for p in sub_pairs if p in vanilla_pairs]
        scr_overlap = [p for p in sub_pairs if p in scratch_pairs]
        triple_overlap_info.append({
            "triple":      triple,
            "van_overlap": van_overlap,
            "scr_overlap": scr_overlap,
        })

    n_triples        = len(scratch_triples)
    n_van_overlap    = sum(1 for t in triple_overlap_info if t["van_overlap"])
    n_no_van_overlap = n_triples - n_van_overlap

    print(f"\n{'─'*65}")
    print(f"Dataset : {ds}")
    print(f"  Vanilla EBM pairs   : {sorted(vanilla_pairs)}")
    print(f"  3-way EBM pairs     : {sorted(scratch_pairs)}")
    print(f"  3-way EBM triples   : {sorted(scratch_triples)}")
    print(f"  # triples with ≥1 Vanilla sub-pair : {n_van_overlap}/{n_triples} "
          f"({100*n_van_overlap/max(n_triples,1):.0f}%)")
    print(f"\n  Triple-level detail:")
    for t in triple_overlap_info:
        van_str = f"Vanilla overlap: {t['van_overlap']}" if t["van_overlap"] else "NO Vanilla overlap"
        print(f"    {t['triple']}  →  {van_str}")

    overlap_rows.append({
        "dataset":           ds,
        "n_vanilla_pairs":   len(vanilla_pairs),
        "n_scratch_pairs":   len(scratch_pairs),
        "n_scratch_triples": n_triples,
        "n_triples_with_vanilla_subpair":    n_van_overlap,
        "n_triples_without_vanilla_subpair": n_no_van_overlap,
        "pct_overlap":       round(100*n_van_overlap/max(n_triples,1), 1),
    })

df_overlap = pd.DataFrame(overlap_rows)
print(f"\n{'='*65}\nSummary:\n")
print(df_overlap.to_string(index=False))
export_table(df_overlap, "Table_TermOverlap")

## BLOCK 14 — Ablation: Cyclic Refitting
Does cyclic refitting improve accuracy? Compare `enable_joint_refit=True` vs `False`.

In [ ]:
print("[BLOCK 14] Ablation: Cyclic Refitting")

# Synthetic + 3 real dataset where Scratch3Way showed strongest gains
ABLATION_DATASETS = DATASETS_SYNTH_MOD + ["cpu_act", "elevators", "naval_propulsion_plant"]

rows_ab = []
for ds in ABLATION_DATASETS:
    X, y, _ = get_data(ds)
    ebm_p   = dict(EBM_BEST[ds])
    for seed in GLOBAL_SEEDS:
        tr, te   = load_split(ds, seed)
        Xtr, ytr = X[tr], y[tr]
        Xte, yte = X[te], y[te]
        for flag in [False, True]:
            m = Sparse3WayEBMWithBagging(
                max_interaction_degree=3, penalty_scale=0.0,
                enable_joint_refit=flag, n_jobs=N_JOBS, **ebm_p)
            print(f"  [{ds}] seed={seed} cyclic={flag} fitting...", flush=True)
            m.fit(Xtr, ytr)
            rows_ab.append({
                "dataset": ds, "seed": seed,
                "cyclic_refit": flag,
                "rmse": rmse(yte, m.predict(Xte))
            })

df_ab     = pd.DataFrame(rows_ab)
df_ab_sum = (df_ab.groupby(["dataset", "cyclic_refit"])
             .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"))
             .reset_index())
display(df_ab_sum)
export_table(df_ab_sum, "Table_Ablation_CyclicRefit")

# ── Plot ──────────────────────────────────────────────────────────────
n_ds = len(ABLATION_DATASETS)
fig, axes = plt.subplots(1, n_ds, figsize=(4*n_ds, 4), sharey=False)
for ax, ds in zip(axes, ABLATION_DATASETS):
    sub = df_ab_sum[df_ab_sum.dataset == ds].sort_values("cyclic_refit")
    ax.plot([0, 1], sub["rmse_mean"].values, marker="o")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["2-stage\n(interpretML)", "Joint\nRefit"])
    ax.set_title(ds); ax.set_ylabel("RMSE (mean)")
fig.suptitle("Ablation: Joint Cyclic Refit vs Two-Stage (interpretML-style)", y=1.02)
plt.tight_layout()
export_fig(fig, "Fig_Ablation_CyclicRefit")
plt.show()

In [ ]:
# ── Pre-experiment: max_bins sensitivity (3-Way EBM, base datasets) ─────────
import time as _t2

_MB_LIST    = [32, 64, 128, 256]
_DATASETS   = ["Friedman1_Mod", "Ishigami_Mod", "Hartmann6D_Mod"]
_SEEDS      = [0, 1, 2]
_N_EST      = 500    
_LR         = 0.05

rows_mb = []
for mb in _MB_LIST:
    for ds in _DATASETS:
        X, y, meta = get_synth_dataset(ds, n_samples=2000, seed=42)
        for seed in _SEEDS:
            tr, te = get_split_indices(len(y), seed)
            m = Sparse3WayEBMWithBagging(
                max_interaction_degree=3,
                penalty_scale=0.0,
                enable_joint_refit=True,
                n_estimators=_N_EST,
                learning_rate=_LR,
                max_bins=mb,
                interactions=3,
                outer_bags=5,
                n_jobs=N_JOBS,
            )
            t0 = _t2.time()
            m.fit(X[tr], y[tr])
            fit_t = _t2.time() - t0
            r = rmse(y[te], m.predict(X[te]))
            rows_mb.append({
                "max_bins": mb, "dataset": ds, "seed": seed,
                "rmse": r, "fit_time": fit_t,
            })
            print(f"  max_bins={mb:<4} | {ds:<18} | seed={seed} | "
                  f"rmse={r:.4f} | {fit_t:.1f}s", flush=True)

df_mb = pd.DataFrame(rows_mb)
df_mb_sum = (df_mb.groupby(["max_bins", "dataset"])
             .agg(rmse_mean=("rmse","mean"),
                  rmse_std =("rmse","std"),
                  fit_mean =("fit_time","mean"))
             .reset_index())

pivot = df_mb_sum.pivot_table(
    index="max_bins", columns="dataset",
    values=["rmse_mean", "rmse_std"]
).round(4)

print("\n=== max_bins Sensitivity — Scratch3Way (mean RMSE, 3 seeds) ===\n")
for ds in _DATASETS:
    print(f"{'max_bins':<10} {'rmse_mean':<12} {'rmse_std'}", end="    ")
print()
print("-" * (len(_DATASETS) * 32))
for mb in _MB_LIST:
    for ds in _DATASETS:
        row = df_mb_sum[(df_mb_sum.max_bins==mb) & (df_mb_sum.dataset==ds)].iloc[0]
        best = df_mb_sum[df_mb_sum.dataset==ds]["rmse_mean"].min()
        marker = " ←" if abs(row.rmse_mean - best) < 1e-6 else "  "
        print(f"{mb:<10} {row.rmse_mean:<12.4f} {row.rmse_std:.4f}{marker}", end="    ")
    print()

export_table(df_mb_sum, "Table_MaxBins_Sensitivity")
print("\n(← = best per dataset)")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FAST-3D Score Distribution Analysis — Friedman1_Mod
# All C(10,3)=120 candidate's S3 score shown, with truth triple highlighted.
# ════════════════════════════════════════════════════════════════════
import itertools, copy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split

_orig_find = Sparse3WayEBM._find_interactions

def _find_interactions_store(self, X, residuals, degree):
    d = X.shape[1]
    top_feats = list(range(d)) if not (d > 20 and degree >= 3) else None
    if top_feats is None:
        feat_agg = np.zeros(d)
        for i, j in itertools.combinations(range(d), 2):
            X_sub = X[:, [i, j]]
            H_s, _ = np.histogramdd(X_sub, bins=8, weights=residuals)
            H_c, _ = np.histogramdd(X_sub, bins=8)
            with np.errstate(divide='ignore', invalid='ignore'):
                means = H_s / H_c
            means[np.isnan(means)] = 0.0
            s = float(np.sum(H_c * means**2))
            feat_agg[i] += s; feat_agg[j] += s
        top_feats = list(np.argsort(feat_agg)[::-1][:20])

    candidates = []
    for combo in itertools.combinations(top_feats, degree):
        X_sub = X[:, combo]
        H_s, _ = np.histogramdd(X_sub, bins=10, weights=residuals)
        H_c, _ = np.histogramdd(X_sub, bins=10)
        with np.errstate(divide='ignore', invalid='ignore'):
            means = H_s / H_c
        means[np.isnan(means)] = 0.0
        candidates.append((float(np.sum(H_c * means**2)), combo))
    candidates.sort(key=lambda x: x[0], reverse=True)
    if degree == 3:
        self.screening_scores_ = candidates   # ← tek ek satır
    return [c[1] for c in candidates[:self.interactions]]

Sparse3WayEBM._find_interactions = _find_interactions_store

# ── Data ─────────────────────────────────────────────────────────────
np.random.seed(42)
X_f, y_f = make_friedman1(n_samples=2000, n_features=10, noise=0.0, random_state=42)
y_f += 25.0 * np.sin(np.pi * X_f[:, 0] * X_f[:, 1] * X_f[:, 2])
X_tr, X_te, y_tr, y_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

# ── Fit (n_estimators=500 enough for screening) ────────────────────
m_screen = Sparse3WayEBM(
    n_estimators=500, learning_rate=0.1, max_bins=32,
    interactions=3, max_interaction_degree=3, penalty_scale=0.0,
    enable_joint_refit=True,   
)
m_screen.bins = []
m_screen.fit(X_tr, y_tr)

# ── Results ─────────────────────────────────────────────────────────
all_scores = m_screen.screening_scores_
TRUTH = (0, 1, 2)
scores = [s for s, _ in all_scores]
combos = [c for _, c in all_scores]
truth_rank = next(r+1 for r,(s,c) in enumerate(all_scores) if set(c)==set(TRUTH))
print(f"Total candidates: {len(all_scores)}, Truth rank: {truth_rank}")
for r,(s,c) in enumerate(all_scores[:5],1):
    print(f"  #{r}: {c}  S3={s:.2f}" + (" ← TRUTH" if set(c)==set(TRUTH) else ""))

# ── Figure ───────────────────────────────────────────────────────────
truth_idx = truth_rank - 1
colors = ['#d62728' if set(c)==set(TRUTH) else '#aec7e8' for c in combos]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, len(scores)+1), scores, color=colors, width=1.0, linewidth=0)
ax.annotate(
    f'Truth: $(x_1,x_2,x_3)$\nRank 1,  $S_3$={scores[truth_idx]:.1f}',
    xy=(truth_idx+1, scores[truth_idx]),
    xytext=(truth_idx+10, scores[truth_idx]*0.80),
    arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.5),
    fontsize=9, color='#d62728'
)
ax.set_xlabel('Candidate Triple (ranked by $S_3$ score)', fontsize=11)
ax.set_ylabel('FAST-3D Score $S_3$', fontsize=11)
ax.set_title('FAST-3D Screening: Score Distribution over All 120 Candidates\n(Friedman1\\_Mod)', fontsize=11)
ax.legend(handles=[
    mpatches.Patch(color='#d62728', label='Truth triple $(x_1,x_2,x_3)$'),
    mpatches.Patch(color='#aec7e8', label='Other candidates (119)')
], fontsize=9, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_fast3d_score_dist.pdf', bbox_inches='tight')
plt.savefig('fig_fast3d_score_dist.png', bbox_inches='tight', dpi=150)
plt.show()


Sparse3WayEBM._find_interactions = _orig_find